In [1]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
from rdkit import Chem
from tqdm import tqdm
import numpy as np
import scipy.stats as ss

import sys
sys.path.append("../framework/code/")
from chemeleon_descriptor import CheMeleonFingerprint

In [11]:
import sklearn; sklearn.__version__

'1.6.1'

In [2]:
with open("oc8b00718_si_002.txt", "r") as f:
    lines = f.readlines()[28:]

smiles = []
mols = []
perms = []
for l in tqdm(lines):
    l = l.rstrip().split()
    smi = l[0]
    try:
        perm = float(l[-1])
    except:
        continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    smiles += [smi]
    mols += [mol]
    perms += [perm]

perms = np.array(perms)

100%|██████████| 92456/92456 [00:01<00:00, 49629.25it/s]


In [3]:
chemeleon_fingerprint = CheMeleonFingerprint()
batch_size, count = 5000, 0
X = []

while count < len(smiles):

    df = chemeleon_fingerprint(smiles[count:count+batch_size])
    X.extend(df)
    count += batch_size

X = np.array(X)

In [4]:
# Set up 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = []

for fold, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = perms[train_index], perms[test_index]

    rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=8)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_scores.append(mse)

    print(f"Fold {fold+1} MSE: {mse:.3f}")

# Overall performance
print(f"\nMean MSE: {np.mean(mse_scores):.3f}")
print(f"Standard Deviation: {np.std(mse_scores):.3f}")

Fold 1 MSE: 0.345
Fold 2 MSE: 0.337
Fold 3 MSE: 0.345
Fold 4 MSE: 0.350
Fold 5 MSE: 0.351

Mean MSE: 0.346
Standard Deviation: 0.005


In [8]:
# plt.scatter(y_pred, y_test, s=3)
# plt.show()

In [6]:
ss.pearsonr(y_pred, y_test)

PearsonRResult(statistic=0.921482161423046, pvalue=0.0)

In [ ]:
rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=8)
rf.fit(X, perms)

In [6]:
y_pred = rf.predict(X)
mse = mean_squared_error(perms, y_pred)
print(f"MSE: {mse:.3f}")

MSE: 0.048


In [7]:
rf

RandomForestRegressor(n_estimators=50, n_jobs=8, random_state=42)

In [9]:
import joblib

# Save the model to a file
joblib.dump(rf, '../checkpoints/RF_REG.joblib')

['../checkpoints/RF_REG.joblib']